In [1]:
from openai import AsyncOpenAI
from copy import deepcopy
import asyncio
import os
import random
import pandas as pd
from tqdm.asyncio import tqdm

In [2]:
openrouter_client = AsyncOpenAI(
    base_url=f"https://openrouter.ai/api/v1",
    api_key="",  # YOUR API KEY
    timeout=60
)

In [3]:
MODEL_NAME = 'openai/gpt-oss-120b:free'

In [4]:
class LLM:
    def __init__(self, use_cache=True, base_model=MODEL_NAME, local_client=openrouter_client):
        self.reset(use_cache, base_model)
        self.lock = asyncio.Lock()
        self.local_client = local_client
    
    def reset(self, use_cache=True, base_model=MODEL_NAME):
        self.cache = {}
        self.use_cache = use_cache
        self.base_model = base_model

    async def _chat_base(self, messages, temperature=0, top_p=1, max_tokens=4096): 
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
            
        cache_id = str(messages) + self.base_model
        
        if cache_id in self.cache and temperature == 0 and self.use_cache:
            return deepcopy(self.cache[cache_id])
            
        response = await self.local_client.chat.completions.create(
            model=self.base_model,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            timeout=60
        )
            
        result_text = response.choices[0].message.content
            
        self.cache[cache_id] = result_text
        return result_text

    async def chat(self, messages, temperature=0, top_p=1, max_tokens=4096):
        return await self._chat_base(messages, temperature, top_p, max_tokens)

    async def chat_serial(self, messages, temperature=0, top_p=1, max_tokens=4096):
        async with self.lock:
            response = await self._chat_base(
                messages,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens
            )

            return response 

In [5]:
async def paraphrase(input_file, output_file):
    llm = LLM()
    df = pd.read_csv(input_file)
    
    if os.path.exists(output_file):
        df_results = pd.read_csv(output_file)
    else:
        df_results = df.copy()
        df_results['paraphrase'] = ""

        df_results = df_results.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    system_prompt = """Ты — лингвист-эксперт. Твоя задача — перефразировать предложение, заменив внутри него идиому на синонимичное неидиоматическое выражение, 
    при этом сохраняя смысл исходного предложения.

    ПРАВИЛА:
    1. В ответе должно строго содержаться перефразированное предложение и не более того.
    2. Перефразирование должно быть неидиоматическим: используй только прямые буквальные формулировки.
    3. Заменяй исключительно идиому, которая указывается в промпте.
    4. Сохраняй структуру предложения, схожую с оригиналом, настолько, насколько это возможно.

    ПРИМЕРЫ:
    1.
    Идиома: бальзам на душу
    Исходное предложение: Эта победа стала для него бальзамом на душу.
    Ответ: Эта победа принесла ему чувство глубокого удовлетворения.

    2.
    Идиома: без понятия
    Исходное предложение: На вопрос о сроках он ответил: «Я без понятия».
    Ответ: На вопрос о сроках он ответил: «Я не знаю».

    3.
    Идиома: бить баклуши
    Исходное предложение: Хватит бить баклуши, пора за работу.
    Ответ: Хватит бездельничать, пора приниматься за работу.
    """

    to_process = df_results[
        (df_results['paraphrase'].isna()) | (df_results['paraphrase'] == "")
    ]

    pbar = tqdm(total=len(df_results), desc="Processing examples")
    pbar.update(len(df_results) - len(to_process))

    for index, row in df_results.iterrows():
        if pd.notna(row['paraphrase']) and row['paraphrase'] != "":
            continue

        idiom = row['idiom']
        context = row['context']

        user_prompt = f"Идиома: {idiom}\nИсходное предложение: {context}\nПерефразируй предложение, заменив указанную идиому."

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    
        try:
            result_raw = await llm.chat(messages)

            if result_raw:
                df_results.at[index, 'paraphrase'] = result_raw
            else:
                df_results.at[index, 'paraphrase'] = "ERROR_EMPTY_RESPONSE"
        
        except Exception as e:
            error_text = str(e)
            print(f"Error at row {index}: {error_text}")

            if "403" in error_text:
                df_results.at[index, 'paraphrase'] = "ERROR_MODERATION"

            else:
                df_results.at[index, 'paraphrase'] = "ERROR_REQUEST"        
        
        pbar.update(1)

        if index % 10 == 0:
            df_results.to_csv(output_file, index=False)

        await asyncio.sleep(random.uniform(2, 5))

    df_results.to_csv(output_file, index=False)

In [ ]:
await paraphrase('ru_idioms_examples_output_not_para.csv', 'ru_idioms_examples_full_pt2_output.csv')